# AI Agente Analyze Fraud Rules 
Mich y cami
Ts
ops_fraud.falcon_declined_transactions fdt
Chargebacks.
Analytics_bi.transactions

User feedback: tabla, declined transactions.

Archivos PLT (v 2.0)

Diario -> dependemos de mich. 
Moverlo a transactions_ytd. 
Agregar 3ds a la tabla


transactions_


Order of Agents to implement
1. Detector de patrón: diario/semanal
- Suggestion: 
- Revise CB 

2. Agente de validación de reglas 
- Sheets - fraud rules


3. Revisión de CB, suggestion de nuevas falcon 
- reglas/limitantes de falcon
- Muy definido el output.
- Use this information de user feedback and suggest a change. 


In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [13]:
from utils.data_ingest import get_db_conn
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format
import gspread
from dateutil.utils import today



In [3]:
import os
import yaml

with open(".config/credentials-mage.yaml", "r") as f:
    creds = yaml.safe_load(f)

for item in creds:
    os.environ[item["name"]] = str(item["value"])

In [4]:
import sys
sys.path.append('/Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space')

In [5]:
import logging

# Set up a basic logger
logger = logging.getLogger("fraud_context")
logger.setLevel(logging.INFO)  

In [6]:
fraud_rules_query = f"""
SELECT user_id, klrid, transaction_id, amount, timestamp_mx_created_at, prosa_timestamp, merchant, mcc_code, regla, pos_entry_mode, pin_capabilities, card_type, product_type
FROM ops_fraud.falcon_declined_transactions;
"""

In [7]:
fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_86335/1460050075.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())


In [8]:
fraud_rules_df.head()

,user_id,klrid,transaction_id,amount,timestamp_mx_created_at,prosa_timestamp,merchant,mcc_code,regla,pos_entry_mode,pin_capabilities,card_type,product_type
0,431c8363-067d-4786-9fa1-e0e7614a5f4f,97ace885-4cd1-4952-938f-49b520a3a17d,PARABILIUM:124767681,-288.99,2025-08-05 00:00:42.912,2025-08-05 00:00:43.144,EBANX 1TIKTOK SH ARTSCIUDAD DE MEXDF MX,7311,VE___Decline_Excessive_Velocity,CNP Manual,unknown,PHYSICAL,CREDIT_5456_BIN
1,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768652,-300.00,2025-08-05 00:07:08.305,2025-08-05 00:07:08.607,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
2,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768690,-300.00,2025-08-05 00:07:24.646,2025-08-05 00:07:24.905,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
3,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768837,-200.00,2025-08-05 00:08:19.412,2025-08-05 00:08:19.657,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
4,0da0f6a0-af55-4fdc-8e12-8a54da213262,73810457-6f0d-463d-abae-2261d89cc2b9,PARABILIUM:124768884,-227.32,2025-08-05 00:08:37.951,2025-08-05 00:08:38.305,D LOCALTDA TEMU MEXICO DF MX,5969,FV___High_Fraud_Score_Online,CNP Manual,unknown,VIRTUAL,CREDIT_5456_BIN


In [9]:
SERVICE_ACCOUNT_FILE = r'.config/klar-cami-cusi.json'
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sheet_id = "1cQ8QzQqml1oBdYliYpW1V2ZquHJrH5iBZzwlGnYFV5I"
spreadsheet = gc.open_by_key(sheet_id)
# connect to ghseets
worksheet = spreadsheet.worksheet('All Falcon Rules')
falcon_rules_df = pd.DataFrame(worksheet.get_all_records())

# PROSA Rules prompt
TODO: 
- explain in prompt what each rule is and what it does,
- explain PROSA dynamics and how it works with the rules, and how the rules are applied in the transactions
- Each rule in the 'All Falcon Rules' worksheet represents a specific fraud prevention rule implement into Falcon, PROSA's fraud detection system. 
- These rules are designed to prevent fraudulent transactions by utilizing various factors and utilizing counter numeric variables.

Each rule has the following characteristics:
- Rule ID: an identifier to group similar rules.
- Rule name: identifier + general description of the rule.
- Logic: detailed rule definition.
- Rule type: the rules usually repeat certain general patterns. 
- Amount limit: if applies for rule, the limit of the amount implemented. 
- Score limit: each transaction in PROSA has a score. If this rule includes a threshold for the score, the limit of the amount implemented.
- Transaction limit: if applies to rule, the limit of how many transactions a user, or list can have. 
- Timeframe: timeeframe for which the rule applies. For example, if a merchant has a card validation merchant of GOOGLE and 24 hours later a TELCEL intent then the transaction does not pass. 
- Lists: There are lists that the rule can use. For example, in the case of MOContador it checks whether a card is inside a archivo de embozo.
- UDV: Each rule has a numeric rule, the numeric rules contain user defined variables that can store numbers (counter), dates, true or False. 
- Segment: The card portfolio group it belongs to.
- Subsegment: subsegment of the segment. 
- Motive: rule was created because of a recent fraud incident, or are general preventative rules.
- Last updated date: date of last update. 
- Creation date: date of rule creation.
- Estado: state of rule. 
- Comments: additional rule comments.

# Tools for pattern detector agent. 

We're going to prioritize the pattern detector agent. We run a weekly analysis of rules and it's motives for declining so, we need to check the following:
- Week by week what are the increases in declinations per rule
- Who are the merchants that are causing these casualites? 
- What's the behavior of the transaction like? Does the user have previous purchaes with this merchant? In velocity cases are they 
    - Is it multiples tries per user and then the trx passes?
    -
- Are there simlar behaviors between the declined transactions? 
- What are pos entry modes of the declined transactions? 
- What segment does the client belong to ?
- What are the stats of the merchant in the last year? last month ? last 7days? Do they significantly differ from the transactions that we declined?
- Does the merchant usually have 3ds, cvv present, not present? 
- Does the merchant currently have CB or active CB? 


# TODO:
- analyze mage analytics infra and see what class we can create for the tools agent manager 

In [10]:
from fraud_utils.ai_agents.fraud_alerts import MerchantFraudContextBuilder

In [11]:
fraud_rules_df["amount_abs"] = fraud_rules_df["amount"].abs()

In [33]:

# Get today's date
today = datetime.today()

# Calculate the start of the current week (Monday as the start of the week)
current_week_start = today - timedelta(days=today.weekday())

# Generate a list of week start dates for the last 7 days and earlier weeks
fraud_rules_df["trx_start_week"] = fraud_rules_df["timestamp_mx_created_at"].apply(
    lambda x: (current_week_start - timedelta(weeks=(current_week_start - x).days // 7)).date()
)
fraud_rules_df["trx_start_week"] = pd.to_datetime(fraud_rules_df["trx_start_week"])

In [26]:
fraud_rules_df[fraud_rules_df["timestamp_mx_created_at"] >= '2026-04-01'].trx_start_week.value_counts()

trx_start_week
2026-04-06    6685
2026-04-13    5456
2026-04-20    4444
2026-05-11    3673
2026-05-04    3059
2026-04-27    2689
2026-05-18    1485
Name: count, dtype: int64

In [55]:
week_analysis = fraud_rules_df.groupby(["trx_start_week", "regla"]).agg(
    transaction_count=("transaction_id", "nunique"),
    total_amount=("amount_abs", "sum")
).reset_index()
# Build a complete week x rule grid so missing weeks are explicit zeros
week_analysis["trx_start_week"] = pd.to_datetime(week_analysis["trx_start_week"]).dt.normalize()

all_weeks = pd.date_range(
    start=week_analysis["trx_start_week"].min(),
    end=week_analysis["trx_start_week"].max(),
    freq="7D"
)
all_rules = week_analysis["regla"].dropna().unique()

full_index = pd.MultiIndex.from_product(
    [all_rules, all_weeks],
    names=["regla", "trx_start_week"]
)

week_analysis = (
    week_analysis
    .set_index(["regla", "trx_start_week"])
    .reindex(full_index, fill_value=0)
    .reset_index()
    .sort_values(["regla", "trx_start_week"])
)

# Week-over-week pct change by rule
pct_cols = week_analysis.groupby("regla")[["transaction_count", "total_amount"]].pct_change().mul(100)
week_analysis["transaction_count_pct_diff"] = pct_cols["transaction_count"]
week_analysis["total_amount_pct_diff"] = pct_cols["total_amount"]

# Clean infinite values from division by zero and round for readability
week_analysis[["transaction_count_pct_diff", "total_amount_pct_diff"]] = (
    week_analysis[["transaction_count_pct_diff", "total_amount_pct_diff"]]
    .round(2)
)


In [68]:
week_observation =  str((current_week_start - timedelta(weeks=1)).date())
week_observation

'2026-05-11'

In [50]:
week_analysis[(week_analysis["trx_start_week"] >= '2026-05-01')
            #    & (week_analysis["transaction_count_pct_diff"] < 0)
               & (week_analysis["regla"] == 'MC_Decline_MCC_Limit_HR')]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2498,MC_Decline_MCC_Limit_HR,2026-05-04,17,"48,738.87",-61.36,-24.81
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2500,MC_Decline_MCC_Limit_HR,2026-05-18,21,"61,674.00",-25.00,37.04


In [58]:
week_analysis = week_analysis[
    (week_analysis.transaction_count_pct_diff.notna()) | (week_analysis.total_amount_pct_diff.notna())
].copy()

In [71]:
week_analysis[
    ((week_analysis["transaction_count_pct_diff"] > 0)
    | (week_analysis["total_amount_pct_diff"] > 0))
    & (week_analysis["trx_start_week"] == week_observation)
]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2827,AV_Decline_Acquirer_CVV_B2C,2026-05-11,9,"9,318.90",inf,inf
2335,FV_Decline_High_Fraud_Score_Online_ALL,2026-05-11,2159,"4,233,311.56",27.53,34.44
367,HF_High_Fraud_Score_General_ALL,2026-05-11,244,"492,090.05",10.41,5.84
3032,KQ_Decline_LumepicKrispy_ALL,2026-05-11,3,568.00,200.00,468.00
2376,LC_Decline_LowScore_CardVerifications_HR,2026-05-11,96,96.00,24.68,24.68
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2868,SN_Decline_Cashout_Validation_B2C,2026-05-11,1,747.09,-50.00,3.54
2950,SQ_Decline_CashOut_Telcel_ALL,2026-05-11,617,"247,010.67",44.16,131.16
1638,VE_Decline_Excessive_Velocity_B2C,2026-05-11,333,"64,318.01",12.50,-12.71
3073,WQ_Decline_WalTaquilla_ALL,2026-05-11,2,"9,113.60",inf,inf


In [73]:
week_analysis[
    ((week_analysis["transaction_count_pct_diff"] > 0)
    | (week_analysis["total_amount_pct_diff"] > 0))
    & (week_analysis["trx_start_week"] == week_observation)
]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2827,AV_Decline_Acquirer_CVV_B2C,2026-05-11,9,"9,318.90",inf,inf
2335,FV_Decline_High_Fraud_Score_Online_ALL,2026-05-11,2159,"4,233,311.56",27.53,34.44
367,HF_High_Fraud_Score_General_ALL,2026-05-11,244,"492,090.05",10.41,5.84
3032,KQ_Decline_LumepicKrispy_ALL,2026-05-11,3,568.00,200.00,468.00
2376,LC_Decline_LowScore_CardVerifications_HR,2026-05-11,96,96.00,24.68,24.68
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2868,SN_Decline_Cashout_Validation_B2C,2026-05-11,1,747.09,-50.00,3.54
2950,SQ_Decline_CashOut_Telcel_ALL,2026-05-11,617,"247,010.67",44.16,131.16
1638,VE_Decline_Excessive_Velocity_B2C,2026-05-11,333,"64,318.01",12.50,-12.71
3073,WQ_Decline_WalTaquilla_ALL,2026-05-11,2,"9,113.60",inf,inf


# Pattern Detector Agent — Weekly Decline Analysis

Analysis of `fraud_rules_df` for the observation week:
1. Top merchants per rule (by volume and amount)
2. Amount similarity across merchants within each rule
3. Peak day of declines per rule

In [76]:
# Filter fraud_rules_df to the observation week only
week_obs_start = pd.Timestamp(week_observation)
week_obs_end = week_obs_start + pd.Timedelta(days=7)
week_obs_end

Timestamp('2026-05-18 00:00:00')

In [85]:

df_week = fraud_rules_df[
    (fraud_rules_df["trx_start_week"] == week_observation)
].copy()

print(f"Observation week: {week_obs_start.date()} → {week_obs_end.date()}")
print(f"Total declined transactions: {df_week['transaction_id'].nunique():,}")
print(f"Total declined amount: ${df_week['amount_abs'].sum():,.2f}")
df_week["regla"].value_counts().head(10)

Observation week: 2026-05-11 → 2026-05-18
Total declined transactions: 3,673
Total declined amount: $5,314,494.47


regla
FV_Decline_High_Fraud_Score_Online_ALL      2159
SQ_Decline_CashOut_Telcel_ALL                617
VE_Decline_Excessive_Velocity_B2C            333
HF_High_Fraud_Score_General_ALL              244
VF_Decline_Velocity_Limit_15mins_B2C         157
LC_Decline_LowScore_CardVerifications_HR      96
MC_Decline_MCC_Limit_HR                       28
MS_Decline_MagneticStripe_Limit_ALL           11
AV_Decline_Acquirer_CVV_B2C                    9
BR_Decline_BL_MAXRISK_PAN_UHR                  8
Name: count, dtype: int64

In [92]:
# 1. Top merchants per rule — by transaction count and total amount
top_merchants_per_rule = (
    df_week
    .groupby(["regla", "merchant"])
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_amount=("amount_abs", "sum"),
        unique_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values(["regla", "transaction_count"], ascending=[True, False])
)


In [93]:
top_merchants_per_rule

,regla,merchant,transaction_count,total_amount,unique_users
2,AV_Decline_Acquirer_CVV_B2C,FERR COMERCIOS ECOM LEON DE LOS A011MX,3,171.43,3
3,AV_Decline_Acquirer_CVV_B2C,VERD CHIKEN BANANA AHOME SIN 025MX,3,992.97,1
1,AV_Decline_Acquirer_CVV_B2C,ENT MOVERESA CUERNAVACA MO017MX,2,"4,654.50",2
0,AV_Decline_Acquirer_CVV_B2C,DONATIVOALEGRIA TOLUCA EM 015MX,1,"3,500.00",1
4,BR_Decline_BL_MAXRISK_PAN_2_UHR,TEMU COM CIUDAD DE MEX001MX,4,4.00,1
...,...,...,...,...,...
987,VF_Decline_Velocity_Limit_15mins_B2C,PAYPAL TECHNOFITNE IZTAPALAPA 00 MX,1,"11,199.00",1
989,VF_Decline_Velocity_Limit_15mins_B2C,SA_fanfills.com Nicosia CY,1,344.04,1
994,VF_Decline_Velocity_Limit_15mins_B2C,T1 TELCEL PYRE CIUDAD DE MEXDF MX,1,50.00,1
995,VF_Decline_Velocity_Limit_15mins_B2C,TEMU COM 1 CIUDAD DE MEX001MX,1,38.79,1


In [ ]:

# Keep top 7 merchants per rule
top_merchants_per_rule = (
    top_merchants_per_rule
    .groupby("regla")
    .apply(lambda x: x.nlargest(7, "transaction_count"))
).reset_index()


In [91]:
top_merchants_per_rule

,regla,level_1,merchant,transaction_count,total_amount,unique_users
0,AV_Decline_Acquirer_CVV_B2C,2,FERR COMERCIOS ECOM LEON DE LOS A011MX,3,171.43,3
1,AV_Decline_Acquirer_CVV_B2C,3,VERD CHIKEN BANANA AHOME SIN 025MX,3,992.97,1
2,AV_Decline_Acquirer_CVV_B2C,1,ENT MOVERESA CUERNAVACA MO017MX,2,"4,654.50",2
3,AV_Decline_Acquirer_CVV_B2C,0,DONATIVOALEGRIA TOLUCA EM 015MX,1,"3,500.00",1
4,BR_Decline_BL_MAXRISK_PAN_2_UHR,4,TEMU COM CIUDAD DE MEX001MX,4,4.00,1
...,...,...,...,...,...,...
60,VF_Decline_Velocity_Limit_15mins_B2C,975,Google TikTok Videos Mountain ViewCA US,8,31.55,2
61,VF_Decline_Velocity_Limit_15mins_B2C,954,CCBill.com OnlyFans Fort LauderdaFL US,7,0.00,5
62,VF_Decline_Velocity_Limit_15mins_B2C,966,GOOGLE TikTok Videos MOUNTAIN VIEWCA US,7,22.96,2
63,VF_Decline_Velocity_Limit_15mins_B2C,988,PAYPAL TEMU 4029357733 00 AU,7,"21,500.39",1


In [81]:
# 2. Amount similarity per merchant within each rule
# Stats: mean, median, std, min, max — a low std relative to mean signals similar amounts (possible fraud pattern)
amount_similarity = (
    df_week
    .groupby(["regla", "merchant"])["amount_abs"]
    .agg(
        trx_count="count",
        mean_amount="mean",
        median_amount="median",
        std_amount="std",
        min_amount="min",
        max_amount="max"
    )
    .reset_index()
)

# Coefficient of variation (std / mean): lower = more uniform amounts
amount_similarity["cv"] = (amount_similarity["std_amount"] / amount_similarity["mean_amount"]).round(3)

amount_similarity = amount_similarity.sort_values(["regla", "trx_count"], ascending=[True, False])
amount_similarity

,regla,merchant,trx_count,mean_amount,median_amount,std_amount,min_amount,max_amount,cv
2,AV_Decline_Acquirer_CVV_B2C,FERR COMERCIOS ECOM LEON DE LOS A011MX,3,57.14,60.00,44.35,11.43,100.00,0.78
3,AV_Decline_Acquirer_CVV_B2C,VERD CHIKEN BANANA AHOME SIN 025MX,3,330.99,330.99,0.00,330.99,330.99,0.00
1,AV_Decline_Acquirer_CVV_B2C,ENT MOVERESA CUERNAVACA MO017MX,2,"2,327.25","2,327.25",40.45,"2,298.65","2,355.85",0.02
0,AV_Decline_Acquirer_CVV_B2C,DONATIVOALEGRIA TOLUCA EM 015MX,1,"3,500.00","3,500.00",NaN,"3,500.00","3,500.00",NaN
4,BR_Decline_BL_MAXRISK_PAN_2_UHR,TEMU COM CIUDAD DE MEX001MX,4,1.00,1.00,0.00,1.00,1.00,0.00
...,...,...,...,...,...,...,...,...,...
987,VF_Decline_Velocity_Limit_15mins_B2C,PAYPAL TECHNOFITNE IZTAPALAPA 00 MX,1,"11,199.00","11,199.00",NaN,"11,199.00","11,199.00",NaN
989,VF_Decline_Velocity_Limit_15mins_B2C,SA_fanfills.com Nicosia CY,1,344.04,344.04,NaN,344.04,344.04,NaN
994,VF_Decline_Velocity_Limit_15mins_B2C,T1 TELCEL PYRE CIUDAD DE MEXDF MX,1,50.00,50.00,NaN,50.00,50.00,NaN
995,VF_Decline_Velocity_Limit_15mins_B2C,TEMU COM 1 CIUDAD DE MEX001MX,1,38.79,38.79,NaN,38.79,38.79,NaN


In [82]:
# 3. Peak day of declines per rule — which day drove the most volume
df_week["trx_day"] = pd.to_datetime(df_week["timestamp_mx_created_at"]).dt.normalize()

peak_day_per_rule = (
    df_week
    .groupby(["regla", "trx_day"])
    .agg(
        transaction_count=("transaction_id", "nunique"),
        total_amount=("amount_abs", "sum")
    )
    .reset_index()
    .sort_values(["regla", "transaction_count"], ascending=[True, False])
)

# Keep only the peak day (top 1) per rule
peak_day_per_rule = (
    peak_day_per_rule
    .groupby("regla", group_keys=False)
    .apply(lambda x: x.nlargest(1, "transaction_count"))
    .reset_index(drop=True)
    .rename(columns={"trx_day": "peak_day"})
)

peak_day_per_rule

,peak_day,transaction_count,total_amount
0,2026-05-05,6,"5,658.90"
1,2026-05-06,4,4.00
2,2026-05-09,3,"1,405.28"
3,2026-05-09,321,"465,338.94"
4,2026-05-05,45,"113,898.99"
5,2026-05-09,2,558.00
6,2026-05-10,19,19.00
7,2026-05-09,9,"26,408.84"
8,2026-05-06,7,"12,758.81"
9,2026-05-11,1,747.09


In [ ]:
fraud_rules_query = f"""
SELECT
    user_id, klrid, transaction_id, amount, timestamp_mx_created_at, prosa_timestamp, merchant, mcc_code, regla, pos_entry_mode, pin_capabilities, card_type, product_type
FROM ops_fraud.falcon_declined_transactions;
"""

NameError: name 'ops_fraud' is not defined

In [97]:
# Tools
full_fraud_query = f""" 
  WITH tc_base AS (
            SELECT
                tc.*,
                COALESCE(
                    NULLIF(split_part(tc.transaction_id, 'PARABILIUM:', 2), ''),
                    tc.transaction_id
                ) AS omi_id_operacion_raw
            FROM ops_fraud.falcon_declined_transactions tc
            WHERE
                tc.timestamp_mx_created_at >= '2026-01-01'
        ),

        oimt_base AS (
            SELECT
                oimt.omi_id_operacion,
                oimt.c063,
                oimt.c032 AS adquirente
            FROM is_pii_parabilium.operation_iso_messages_temp oimt
            INNER JOIN tc_base tc
                ON oimt.omi_id_operacion = tc.omi_id_operacion_raw
        ),

        pin_verif AS (
            SELECT
                o.omi_id_operacion,
                '! ' || REGEXP_SUBSTR(o.c063, 'B300080[^!]*') AS B300080_value,
                SUBSTRING(B300080_value FROM 49 FOR 6) AS "8-CVMRSLTS",
                SUBSTRING("8-CVMRSLTS" FROM 1 FOR 2) AS byte_1_hex,
                CASE
                    WHEN byte_1_hex ~ '^[0-9A-Fa-f]{2}$'
                    THEN FROM_VARBYTE(from_hex(byte_1_hex), 'binary')
                    ELSE NULL
                END AS byte_1_bits,
                CASE
                    WHEN byte_1_bits IS NULL THEN NULL
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000000'
                        THEN 'Procesamiento de CVM fallido'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000001'
                        THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000010'
                        THEN 'PIN cifrado verificado en linea'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000011'
                        THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta y firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000100'
                        THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000101'
                        THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta y firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011110'
                        THEN 'Firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011111'
                        THEN 'No se requiere CVM'
                    ELSE SUBSTRING(byte_1_bits FROM 3 FOR 6)
                END AS metodo_verificacion
            FROM oimt_base o
            WHERE o.c063 LIKE '%! B3%'
        ),

        three_ds AS (
            SELECT
                o.omi_id_operacion,
                o.c063,
                REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') AS ce_token,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NOT NULL
                        AND POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) > 0
                    THEN SUBSTRING(
                        REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                        FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                        FOR 2
                    )
                    ELSE NULL
                END AS leading_indicator,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
                        THEN 'NO_3DS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kA','kB','kC','kE','kF','kJ','kR','kS','kG','kO','kP')
                        THEN '3DS_AUTHENTICATED'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kN','kW','kU','kX')
                        THEN '3DS_NOT_AUTHENTICATED'
                    ELSE 'UNKNOWN'
                END AS three_ds_status,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
                        THEN 'NO_3DS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kA','kC','kE','kO')
                        THEN 'FRICTIONLESS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kB','kS','kG','kP')
                        THEN 'CHALLENGE'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kN','kW','kU','kX')
                        THEN 'EXEMPT_OR_INFO'
                    ELSE 'UNKNOWN'
                END AS three_ds_flow
            FROM oimt_base o
        )

        SELECT
            tc.*,
            pv.metodo_verificacion AS metodo_identificacion,
            td.leading_indicator,
            td.three_ds_status,
            td.three_ds_flow,
            td.c063,
            o.sucursal AS afiliacion,
            o.terminal AS numero_terminal,
            ob.adquirente
        FROM tc_base tc
        LEFT JOIN pin_verif pv
            ON pv.omi_id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN three_ds td
            ON td.omi_id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN is_pii_parabilium.parabilium_transactions pt
            ON pt.id = tc.omi_id_operacion_raw
        LEFT JOIN is_pii_parabilium.operaciones o
            ON o.id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN oimt_base ob
            ON ob.omi_id_operacion = tc.omi_id_operacion_raw
    """

In [98]:
df = pd.read_sql_query(full_fraud_query, get_db_conn())

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_86335/1738825921.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(full_fraud_query, get_db_conn())


In [99]:
df.head()

,user_id,klrid,transaction_id,amount,timestamp_mx_created_at,prosa_timestamp,merchant,mcc_code,regla,pos_entry_mode,pin_capabilities,card_type,product_type,omi_id_operacion_raw,metodo_identificacion,leading_indicator,three_ds_status,three_ds_flow,c063,afiliacion,numero_terminal,adquirente
0,9a2ee3f3-1dd6-440a-b390-c5cd6b76250a,8449d0cb-abbc-480b-a471-712c2a7d1103,176602545,-38.00,2026-01-01 01:35:38.322,2026-01-01 01:35:38.531,GOOGLE CR CIUDAD DE MEX001MX,7311,GO_Velocity_Limit_Google_B2C,CNP Manual,unknown,VIRTUAL,CREDIT_5401_BIN,176602545,NaN,NaN,UNKNOWN,UNKNOWN,& 0000500112! C000026 001 113207 0 0...,9571637,CPE 1,544550
1,8242450e-e934-4ab6-8205-8e39442f4cba,5e5f623d-8a37-4548-b75b-b1079d20a004,PARABILIUM:176622718,"-2,178.48",2026-01-01 04:50:21.779,2026-01-01 04:50:21.989,SERVICIOS DIGITALES 1 CIUDAD DE MEX001MX,7399,FV_High_Fraud_Score_Online_ALL_EXCEPT_RISKY_PAN,CNP Manual,unknown,VIRTUAL,CREDIT_5456_BIN,176622718,NaN,NaN,UNKNOWN,UNKNOWN,& 0000500112! C000026 001 113207 0 0...,9265986,MEDIO,544550
2,94060f05-e0f8-4d9b-a6e7-4cf94cd588df,ac6f7166-b97d-434e-a2ff-d64bca9f1359,PARABILIUM:176740141,-1.00,2026-01-01 15:55:38.316,2026-01-01 15:55:38.666,OPENPAYDIDI CIUDAD DE MEXCMXMX,4789,FV_High_Fraud_Score_Online_ALL_EXCEPT_RISKY_PAN,CNP Manual,cannot accept pin,PHYSICAL,CREDIT_5401_BIN,176740141,NaN,NaN,UNKNOWN,UNKNOWN,& 0000400082! C000026 XXX 00106760 7 1 0...,4512355,00009887,12
3,5504d34f-bac7-4775-adfc-c719da388f30,8c64a1d4-f295-4358-bfab-7bf2a10e4c6b,PARABILIUM:176791254,-625.69,2026-01-01 18:50:50.107,2026-01-01 18:50:50.375,TEMU COM 1 CIUDAD DE MEX001MX,7399,FV_High_Fraud_Score_Online_ALL_EXCEPT_RISKY_PAN,CNP Manual,unknown,PHYSICAL,CREDIT_5401_BIN,176791254,NaN,NaN,UNKNOWN,UNKNOWN,& 0000500112! C000026 001 113207 0 0...,9048449,SITIO 1,544550
4,71a50db4-09d3-4c1d-b93c-e7e8592f11c7,7b68b52b-b779-465d-8513-83d03bdb810e,176831445,-1.49,2026-01-01 21:12:54.396,2026-01-01 21:12:54.572,Google TikTok Videos Mountain ViewCA US,5816,GO_Velocity_Limit_Google_B2C,CNP Card On File,cannot accept pin,PHYSICAL,CREDIT_5401_BIN,176831445,NaN,NaN,UNKNOWN,UNKNOWN,& 0000600228! C000026 94043 7 0...,5270210,014021,14021


In [100]:
today = datetime.today()

# Calculate the start of the current week (Monday as the start of the week)
current_week_start = today - timedelta(days=today.weekday())

# Generate a list of week start dates for the last 7 days and earlier weeks
df["trx_start_week"] = df["timestamp_mx_created_at"].apply(
    lambda x: (current_week_start - timedelta(weeks=(current_week_start - x).days // 7)).date()
)
df["trx_start_week"] = pd.to_datetime(df["trx_start_week"])


# Last 7 Days Decline Analysis

Uses the pandas DataFrame `df` to compare the last 7 complete days against the previous 7 days. The outputs answer:
- amount declined, transactions declined, and users declined to
- increase/decrease versus the week before
- the same metrics by `product_type`, `card_type`, `pos_entry_mode`, `mcc_code`, and transaction hour

In [110]:
declines_df = df.copy()

required_columns = [
    "timestamp_mx_created_at",
    "transaction_id",
    "user_id",
    "amount",
    "product_type",
    "card_type",
    "pos_entry_mode",
    "mcc_code",
]
missing_columns = [col for col in required_columns if col not in declines_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in df: {missing_columns}")

declines_df["timestamp_mx_created_at"] = pd.to_datetime(
    declines_df["timestamp_mx_created_at"],
    errors="coerce",
)
declines_df["amount_declined_abs"] = pd.to_numeric(
    declines_df["amount"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
).abs().fillna(0)
declines_df = declines_df[declines_df["timestamp_mx_created_at"].notna()].copy()

# Last 7 complete days, excluding the current partial day.
analysis_end = pd.Timestamp.now(tz="America/Mexico_City").tz_localize(None).normalize()
last_7_start = analysis_end - pd.Timedelta(days=8)
previous_7_start = last_7_start - pd.Timedelta(days=7)

declines_last_7d = declines_df[
    (declines_df["timestamp_mx_created_at"] >= last_7_start)
    & (declines_df["timestamp_mx_created_at"] < analysis_end)
].copy()

declines_previous_7d = declines_df[
    (declines_df["timestamp_mx_created_at"] >= previous_7_start)
    & (declines_df["timestamp_mx_created_at"] < last_7_start)
].copy()

print(f"Last 7 complete days: {last_7_start:%Y-%m-%d} to {analysis_end:%Y-%m-%d} (exclusive)")
print(f"Previous 7 days: {previous_7_start:%Y-%m-%d} to {last_7_start:%Y-%m-%d} (exclusive)")
print(f"Rows in last 7 days: {len(declines_last_7d):,}")
print(f"Rows in previous 7 days: {len(declines_previous_7d):,}")

Last 7 complete days: 2026-05-10 to 2026-05-18 (exclusive)
Previous 7 days: 2026-05-03 to 2026-05-10 (exclusive)
Rows in last 7 days: 2,249
Rows in previous 7 days: 3,856


In [111]:
def decline_metrics(input_df: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "transactions_declined": input_df["transaction_id"].nunique(),
            "amount_declined": input_df["amount_declined_abs"].sum(),
            "users_declined": input_df["user_id"].nunique(),
        }
    )


def safe_pct_change(current: pd.Series, previous: pd.Series) -> pd.Series:
    return np.where(
        previous.eq(0),
        np.where(current.eq(0), 0, np.nan),
        ((current - previous) / previous) * 100,
    )


last_7_metrics = decline_metrics(declines_last_7d)
previous_7_metrics = decline_metrics(declines_previous_7d)

decline_period_summary = pd.DataFrame(
    [
        {
            "period": "previous_7_days",
            "start_date": previous_7_start.date(),
            "end_date_exclusive": last_7_start.date(),
            **previous_7_metrics.to_dict(),
        },
        {
            "period": "last_7_days",
            "start_date": last_7_start.date(),
            "end_date_exclusive": analysis_end.date(),
            **last_7_metrics.to_dict(),
        },
    ]
)

overall_week_over_week = pd.DataFrame(
    {
        "metric": last_7_metrics.index,
        "last_7_days": last_7_metrics.values,
        "previous_7_days": previous_7_metrics.values,
    }
)
overall_week_over_week["absolute_change"] = (
    overall_week_over_week["last_7_days"] - overall_week_over_week["previous_7_days"]
)
overall_week_over_week["pct_change"] = safe_pct_change(
    overall_week_over_week["last_7_days"],
    overall_week_over_week["previous_7_days"],
)
overall_week_over_week["change_direction"] = np.select(
    [
        overall_week_over_week["absolute_change"] > 0,
        overall_week_over_week["absolute_change"] < 0,
    ],
    ["increase", "decrease"],
    default="no_change",
)


In [113]:
# general analysis
print('Overall week-over-week decline metrics:')
overall_week_over_week[["metric" , "last_7_days", "pct_change", "change_direction"]]

Overall week-over-week decline metrics:


,metric,last_7_days,pct_change,change_direction
0,transactions_declined,"2,249.00",-41.68,decrease
1,amount_declined,"3,150,954.36",-40.56,decrease
2,users_declined,765.00,-46.24,decrease


## Breakdowns by Product, Card, POS, MCC, and Hour

`dimension_breakdowns` stores one DataFrame per dimension. `all_dimension_breakdowns` combines all dimensions into one table for ranking the biggest movements.

In [126]:
dimension_breakdowns["card_type"]

,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,card_type,PHYSICAL,1229,"1,634,289.76",388,1818,"2,575,910.95",737,-589,-32.40,decrease,"-941,621.19",-36.55,decrease,-349,-47.35,decrease,False
1,card_type,VIRTUAL,1020,"1,516,664.60",383,2038,"2,724,783.12",696,-1018,-49.95,decrease,"-1,208,118.52",-44.34,decrease,-313,-44.97,decrease,False


In [139]:
declines_last_7d["transaction_hour"] = declines_last_7d["timestamp_mx_created_at"].dt.hour
declines_previous_7d["transaction_hour"] = declines_previous_7d["timestamp_mx_created_at"].dt.hour

breakdown_dimensions = [
    "product_type",
    "card_type",
    "pos_entry_mode",
    "regla",
    "mcc_code",
    "transaction_hour",
    
]


def dimension_metrics(input_df: pd.DataFrame, dimension: str) -> pd.DataFrame:
    grouped_df = input_df.copy()
    grouped_df[dimension] = grouped_df[dimension].fillna("UNKNOWN").astype(str)

    return (
        grouped_df.groupby(dimension, dropna=False)
        .agg(
            transactions_declined=("transaction_id", "nunique"),
            amount_declined=("amount_declined_abs", "sum"),
            users_declined=("user_id", "nunique"),
        )
        .reset_index()
        .rename(columns={dimension: "dimension_value"})
    )


def build_decline_breakdown(dimension: str) -> pd.DataFrame:
    current = dimension_metrics(declines_last_7d, dimension).rename(
        columns={
            "transactions_declined": "last_7_transactions_declined",
            "amount_declined": "last_7_amount_declined",
            "users_declined": "last_7_users_declined",
        }
    )
    previous = dimension_metrics(declines_previous_7d, dimension).rename(
        columns={
            "transactions_declined": "previous_7_transactions_declined",
            "amount_declined": "previous_7_amount_declined",
            "users_declined": "previous_7_users_declined",
        }
    )

    breakdown = current.merge(previous, on="dimension_value", how="outer").fillna(0)
    breakdown.insert(0, "dimension", dimension)

    metrics = ["transactions_declined", "amount_declined", "users_declined"]
    for metric in metrics:
        current_col = f"last_7_{metric}"
        previous_col = f"previous_7_{metric}"
        breakdown[f"{metric}_change"] = breakdown[current_col] - breakdown[previous_col]
        breakdown[f"{metric}_pct_change"] = safe_pct_change(
            breakdown[current_col],
            breakdown[previous_col],
        )
        breakdown[f"{metric}_change_direction"] = np.select(
            [
                breakdown[f"{metric}_change"] > 0,
                breakdown[f"{metric}_change"] < 0,
            ],
            ["increase", "decrease"],
            default="no_change",
        )

    breakdown["new_in_last_7_days"] = (
        (breakdown["last_7_transactions_declined"] > 0)
        & (breakdown["previous_7_transactions_declined"] == 0)
    )

    return breakdown.sort_values(
        ["last_7_transactions_declined", "transactions_declined_change"],
        ascending=False,
    ).reset_index(drop=True)


dimension_breakdowns = {
    dimension: build_decline_breakdown(dimension)
    for dimension in breakdown_dimensions
}

all_dimension_breakdowns = pd.concat(
    dimension_breakdowns.values(),
    ignore_index=True,
)

product_type_decline_breakdown = dimension_breakdowns["product_type"]
card_type_decline_breakdown = dimension_breakdowns["card_type"]
pos_entry_mode_decline_breakdown = dimension_breakdowns["pos_entry_mode"]
mcc_code_decline_breakdown = dimension_breakdowns["mcc_code"]
transaction_hour_decline_breakdown = dimension_breakdowns["transaction_hour"]

all_dimension_breakdowns.head(25)

,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,product_type,CREDIT_5401_BIN,"1,917.00","1,876,474.67",628.00,"3,206.00","3,547,159.45","1,179.00","-1,289.00",-40.21,decrease,"-1,670,684.78",-47.10,decrease,-551.00,-46.73,decrease,False
1,product_type,CREDIT_5456_BIN,173.00,"465,441.54",72.00,345.00,"517,536.25",134.00,-172.00,-49.86,decrease,"-52,094.71",-10.07,decrease,-62.00,-46.27,decrease,False
2,product_type,PLATINUM,158.00,"808,938.15",64.00,287.00,"1,188,067.12",108.00,-129.00,-44.95,decrease,"-379,128.97",-31.91,decrease,-44.00,-40.74,decrease,False
3,product_type,BUSINESS_NEW_BIN,1.00,100.00,1.00,17.00,"40,881.25",3.00,-16.00,-94.12,decrease,"-40,781.25",-99.76,decrease,-2.00,-66.67,decrease,False
4,product_type,UBER_CREDIT_BIN,0.00,0.00,0.00,1.00,"7,050.00",1.00,-1.00,-100.00,decrease,"-7,050.00",-100.00,decrease,-1.00,-100.00,decrease,False
5,card_type,PHYSICAL,"1,229.00","1,634,289.76",388.00,"1,818.00","2,575,910.95",737.00,-589.00,-32.40,decrease,"-941,621.19",-36.55,decrease,-349.00,-47.35,decrease,False
6,card_type,VIRTUAL,"1,020.00","1,516,664.60",383.00,"2,038.00","2,724,783.12",696.00,"-1,018.00",-49.95,decrease,"-1,208,118.52",-44.34,decrease,-313.00,-44.97,decrease,False
7,pos_entry_mode,CNP Manual,"1,488.00","2,390,324.49",631.00,"2,936.00","4,257,963.65","1,234.00","-1,448.00",-49.32,decrease,"-1,867,639.16",-43.86,decrease,-603.00,-48.87,decrease,False
8,pos_entry_mode,CNP Card On File,716.00,"669,674.94",148.00,832.00,"915,333.17",226.00,-116.00,-13.94,decrease,"-245,658.23",-26.84,decrease,-78.00,-34.51,decrease,False
9,pos_entry_mode,Contactless,21.00,"32,673.45",11.00,31.00,"56,641.04",19.00,-10.00,-32.26,decrease,"-23,967.59",-42.31,decrease,-8.00,-42.11,decrease,False


In [136]:
all_dimension_breakdowns["dimension"].unique()

<StringArray>
['product_type', 'card_type', 'pos_entry_mode', 'mcc_code',
 'transaction_hour']
Length: 5, dtype: str

In [ ]:
all_dimension_breakdowns[
    ~(all_dimension_breakdowns["dimension"].isin(['mcc_code','transaction_hour']))
    & ((all_dimension_breakdowns["transactions_declined_change"] >= 0)
    |(all_dimension_breakdowns["amount_declined_pct_change"] >= 0)
        |(all_dimension_breakdowns["users_declined_pct_change"] >= 0)
    )
][['dimension','dimension_value','last_7_transactions_declined', 'last_7_amount_declined', 'last_7_users_declined', 'transactions_declined_pct_change', 'amount_declined_pct_change', 'users_declined_pct_change']]

,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,transactions_declined_pct_change,amount_declined_pct_change,users_declined_pct_change
10,pos_entry_mode,Magnetic stripe read,11.00,"37,917.52",3.00,-47.62,107.52,-25.00
19,regla,MC_Decline_MCC_Limit_HR,23.00,"69,677.00",9.00,-28.12,26.24,-25.00
20,regla,MS_Decline_MagneticStripe_Limit_ALL,13.00,"53,759.74",5.00,-38.10,40.50,-16.67
22,regla,SN_Decline_Cashout_Validation_B2C,1.00,747.09,1.00,0.00,99.81,0.00
23,regla,KQ_Decline_LumepicKrispy_ALL,1.00,10.00,1.00,-50.00,-98.21,0.00


In [ ]:
# all time metrics per rules


In [115]:
top_decline_increase_drivers = all_dimension_breakdowns.sort_values(
    [
        "transactions_declined_change",
        "amount_declined_change",
        "users_declined_change",
    ],
    ascending=False,
).head(25)

top_decline_increase_drivers

,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
13,mcc_code,7311,475.00,"181,210.49",99.00,278.00,"460,484.91",124.00,197.00,70.86,increase,"-279,274.42",-60.65,decrease,-25.00,-20.16,decrease,False
17,mcc_code,7941,125.00,"56,195.88",20.00,82.00,"55,886.00",27.00,43.00,52.44,increase,309.88,0.55,increase,-7.00,-25.93,decrease,False
34,mcc_code,4511,18.00,"58,627.29",8.00,4.00,"98,076.06",4.00,14.00,350.00,increase,"-39,448.77",-40.22,decrease,4.00,100.00,increase,False
23,mcc_code,4899,40.00,"3,330.58",8.00,30.00,"6,810.20",15.00,10.00,33.33,increase,"-3,479.62",-51.09,decrease,-7.00,-46.67,decrease,False
46,mcc_code,5533,9.00,"40,026.24",2.00,0.00,0.00,0.00,9.00,NaN,increase,"40,026.24",NaN,increase,2.00,NaN,increase,True
41,mcc_code,7841,11.00,"16,732.15",1.00,4.00,824.64,2.00,7.00,175.00,increase,"15,907.51","1,929.02",increase,-1.00,-50.00,decrease,False
158,transaction_hour,4,57.00,"36,169.14",16.00,51.00,"38,793.08",21.00,6.00,11.76,increase,"-2,623.94",-6.76,decrease,-5.00,-23.81,decrease,False
47,mcc_code,5039,9.00,"48,246.59",5.00,4.00,"5,310.77",4.00,5.00,125.00,increase,"42,935.82",808.47,increase,1.00,25.00,increase,False
53,mcc_code,5691,6.00,"17,614.38",3.00,1.00,"2,653.00",1.00,5.00,500.00,increase,"14,961.38",563.94,increase,2.00,200.00,increase,False
58,mcc_code,8699,5.00,75.91,2.00,0.00,0.00,0.00,5.00,NaN,increase,75.91,NaN,increase,2.00,NaN,increase,True


In [116]:
top_decline_decrease_drivers = all_dimension_breakdowns.sort_values(
    [
        "transactions_declined_change",
        "amount_declined_change",
        "users_declined_change",
    ],
    ascending=True,
).head(25)

top_decline_decrease_drivers

,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
7,pos_entry_mode,CNP Manual,"1,488.00","2,390,324.49",631.00,"2,936.00","4,257,963.65","1,234.00","-1,448.00",-49.32,decrease,"-1,867,639.16",-43.86,decrease,-603.00,-48.87,decrease,False
0,product_type,CREDIT_5401_BIN,"1,917.00","1,876,474.67",628.00,"3,206.00","3,547,159.45","1,179.00","-1,289.00",-40.21,decrease,"-1,670,684.78",-47.10,decrease,-551.00,-46.73,decrease,False
6,card_type,VIRTUAL,"1,020.00","1,516,664.60",383.00,"2,038.00","2,724,783.12",696.00,"-1,018.00",-49.95,decrease,"-1,208,118.52",-44.34,decrease,-313.00,-44.97,decrease,False
5,card_type,PHYSICAL,"1,229.00","1,634,289.76",388.00,"1,818.00","2,575,910.95",737.00,-589.00,-32.40,decrease,"-941,621.19",-36.55,decrease,-349.00,-47.35,decrease,False
15,mcc_code,4814,210.00,"57,390.25",81.00,586.00,"193,758.54",221.00,-376.00,-64.16,decrease,"-136,368.29",-70.38,decrease,-140.00,-63.35,decrease,False
14,mcc_code,7399,248.00,"219,493.40",101.00,477.00,"359,463.57",187.00,-229.00,-48.01,decrease,"-139,970.17",-38.94,decrease,-86.00,-45.99,decrease,False
21,mcc_code,5967,50.00,"20,571.20",18.00,269.00,"95,185.79",48.00,-219.00,-81.41,decrease,"-74,614.59",-78.39,decrease,-30.00,-62.50,decrease,False
16,mcc_code,5399,172.00,"377,237.04",111.00,384.00,"1,074,967.51",229.00,-212.00,-55.21,decrease,"-697,730.47",-64.91,decrease,-118.00,-51.53,decrease,False
1,product_type,CREDIT_5456_BIN,173.00,"465,441.54",72.00,345.00,"517,536.25",134.00,-172.00,-49.86,decrease,"-52,094.71",-10.07,decrease,-62.00,-46.27,decrease,False
149,transaction_hour,17,85.00,"173,974.74",48.00,246.00,"314,869.13",111.00,-161.00,-65.45,decrease,"-140,894.39",-44.75,decrease,-63.00,-56.76,decrease,False


In [117]:
# Run this cell to inspect the top rows for every dimension separately.
for dimension, breakdown in dimension_breakdowns.items():
    print(f"\n{dimension}")
    display(breakdown.head(20))


product_type


,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,product_type,CREDIT_5401_BIN,"1,917.00","1,876,474.67",628.00,3206,"3,547,159.45",1179,"-1,289.00",-40.21,decrease,"-1,670,684.78",-47.10,decrease,-551.00,-46.73,decrease,False
1,product_type,CREDIT_5456_BIN,173.00,"465,441.54",72.00,345,"517,536.25",134,-172.00,-49.86,decrease,"-52,094.71",-10.07,decrease,-62.00,-46.27,decrease,False
2,product_type,PLATINUM,158.00,"808,938.15",64.00,287,"1,188,067.12",108,-129.00,-44.95,decrease,"-379,128.97",-31.91,decrease,-44.00,-40.74,decrease,False
3,product_type,BUSINESS_NEW_BIN,1.00,100.00,1.00,17,"40,881.25",3,-16.00,-94.12,decrease,"-40,781.25",-99.76,decrease,-2.00,-66.67,decrease,False
4,product_type,UBER_CREDIT_BIN,0.00,0.00,0.00,1,"7,050.00",1,-1.00,-100.00,decrease,"-7,050.00",-100.00,decrease,-1.00,-100.00,decrease,False



card_type


,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,card_type,PHYSICAL,1229,"1,634,289.76",388,1818,"2,575,910.95",737,-589,-32.40,decrease,"-941,621.19",-36.55,decrease,-349,-47.35,decrease,False
1,card_type,VIRTUAL,1020,"1,516,664.60",383,2038,"2,724,783.12",696,-1018,-49.95,decrease,"-1,208,118.52",-44.34,decrease,-313,-44.97,decrease,False



pos_entry_mode


,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,pos_entry_mode,CNP Manual,1488,"2,390,324.49",631,2936,"4,257,963.65",1234,-1448,-49.32,decrease,"-1,867,639.16",-43.86,decrease,-603,-48.87,decrease,False
1,pos_entry_mode,CNP Card On File,716,"669,674.94",148,832,"915,333.17",226,-116,-13.94,decrease,"-245,658.23",-26.84,decrease,-78,-34.51,decrease,False
2,pos_entry_mode,Contactless,21,"32,673.45",11,31,"56,641.04",19,-10,-32.26,decrease,"-23,967.59",-42.31,decrease,-8,-42.11,decrease,False
3,pos_entry_mode,Magnetic stripe read,11,"37,917.52",3,21,"18,271.44",4,-10,-47.62,decrease,"19,646.08",107.52,increase,-1,-25.00,decrease,False
4,pos_entry_mode,Integrated Circuit Read,11,"4,521.74",6,33,"32,330.29",21,-22,-66.67,decrease,"-27,808.55",-86.01,decrease,-15,-71.43,decrease,False
5,pos_entry_mode,Fallback,2,"15,842.22",2,3,"20,154.48",3,-1,-33.33,decrease,"-4,312.26",-21.40,decrease,-1,-33.33,decrease,False



mcc_code


,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,mcc_code,7311,475.00,"181,210.49",99.00,278.00,"460,484.91",124.00,197.00,70.86,increase,"-279,274.42",-60.65,decrease,-25.00,-20.16,decrease,False
1,mcc_code,7399,248.00,"219,493.40",101.00,477.00,"359,463.57",187.00,-229.00,-48.01,decrease,"-139,970.17",-38.94,decrease,-86.00,-45.99,decrease,False
2,mcc_code,4814,210.00,"57,390.25",81.00,586.00,"193,758.54",221.00,-376.00,-64.16,decrease,"-136,368.29",-70.38,decrease,-140.00,-63.35,decrease,False
3,mcc_code,5399,172.00,"377,237.04",111.00,384.00,"1,074,967.51",229.00,-212.00,-55.21,decrease,"-697,730.47",-64.91,decrease,-118.00,-51.53,decrease,False
4,mcc_code,7941,125.00,"56,195.88",20.00,82.00,"55,886.00",27.00,43.00,52.44,increase,309.88,0.55,increase,-7.00,-25.93,decrease,False
5,mcc_code,4121,85.00,"93,598.68",39.00,114.00,"221,027.75",61.00,-29.00,-25.44,decrease,"-127,429.07",-57.65,decrease,-22.00,-36.07,decrease,False
6,mcc_code,4816,74.00,"21,950.98",30.00,214.00,"61,205.92",124.00,-140.00,-65.42,decrease,"-39,254.94",-64.14,decrease,-94.00,-75.81,decrease,False
7,mcc_code,5816,72.00,"23,216.52",35.00,198.00,"51,849.61",68.00,-126.00,-63.64,decrease,"-28,633.09",-55.22,decrease,-33.00,-48.53,decrease,False
8,mcc_code,5967,50.00,"20,571.20",18.00,269.00,"95,185.79",48.00,-219.00,-81.41,decrease,"-74,614.59",-78.39,decrease,-30.00,-62.50,decrease,False
9,mcc_code,5818,49.00,"9,323.69",13.00,74.00,"22,921.60",29.00,-25.00,-33.78,decrease,"-13,597.91",-59.32,decrease,-16.00,-55.17,decrease,False



transaction_hour


,dimension,dimension_value,last_7_transactions_declined,last_7_amount_declined,last_7_users_declined,previous_7_transactions_declined,previous_7_amount_declined,previous_7_users_declined,transactions_declined_change,transactions_declined_pct_change,transactions_declined_change_direction,amount_declined_change,amount_declined_pct_change,amount_declined_change_direction,users_declined_change,users_declined_pct_change,users_declined_change_direction,new_in_last_7_days
0,transaction_hour,23,169,"455,292.96",52,214,"226,661.10",89,-45,-21.03,decrease,"228,631.86",100.87,increase,-37,-41.57,decrease,False
1,transaction_hour,0,157,"115,824.37",48,180,"200,161.21",80,-23,-12.78,decrease,"-84,336.84",-42.13,decrease,-32,-40.00,decrease,False
2,transaction_hour,18,130,"171,066.78",56,191,"354,477.86",100,-61,-31.94,decrease,"-183,411.08",-51.74,decrease,-44,-44.00,decrease,False
3,transaction_hour,13,125,"203,937.57",64,208,"309,847.15",107,-83,-39.90,decrease,"-105,909.58",-34.18,decrease,-43,-40.19,decrease,False
4,transaction_hour,22,123,"98,734.40",50,192,"161,305.68",87,-69,-35.94,decrease,"-62,571.28",-38.79,decrease,-37,-42.53,decrease,False
5,transaction_hour,19,123,"305,476.10",50,227,"499,976.50",97,-104,-45.81,decrease,"-194,500.40",-38.90,decrease,-47,-48.45,decrease,False
6,transaction_hour,11,120,"113,648.93",49,167,"231,122.09",90,-47,-28.14,decrease,"-117,473.16",-50.83,decrease,-41,-45.56,decrease,False
7,transaction_hour,21,104,"153,318.12",54,191,"231,806.47",93,-87,-45.55,decrease,"-78,488.35",-33.86,decrease,-39,-41.94,decrease,False
8,transaction_hour,12,103,"143,957.35",60,212,"483,186.69",101,-109,-51.42,decrease,"-339,229.34",-70.21,decrease,-41,-40.59,decrease,False
9,transaction_hour,15,99,"158,132.90",49,180,"215,861.51",98,-81,-45.00,decrease,"-57,728.61",-26.74,decrease,-49,-50.00,decrease,False
